# 17. Putting Everything Together

## Problem

如果把前面所有模块真正拼回一张图，一个现代 LLM 到底由哪些部分组成？这些部分分别解决什么问题？DeepSeek-like 模型为什么既要讲结构，也要讲数据和多阶段训练？

## Dependencies

建议完整读过前面关于 tokenizer、attention、GQA、MLA、MoE、pretraining、SFT、reward model 的 notebook。


## Three lines you now need to hold at once

走到这里，真正重要的不是再多记一个名词，而是把三条线同时放进脑子里：

1. **数据线**：文本从哪里来，如何爬取、清洗、去重、过滤。
2. **模型线**：token 如何变成 embedding，attention 如何建模关系，GQA / MLA / MoE 如何做效率和容量优化。
3. **训练线**：pretraining、continued training、SFT、reward model、RLHF / PPO / GRPO 如何一步步塑造能力和行为。

前面如果只学单点，这三条线会分散。到了这里，目标就是把它们重新拉成一个闭环。


## A complete mental map

你可以把一个现代 LLM 粗略拆成下面几层：

### 1. Data layer

这里回答的是“模型吃什么”。

- 原始网页、代码、文档、对话、题库、论坛内容先被收集。
- 之后做正文抽取、去重、质量过滤、分桶。
- 最终得到适合预训练或对齐训练的数据集。

### 2. Representation layer

这里回答的是“原始文本如何进入网络”。

- tokenizer 把字符串切成 token ids。
- embedding 把离散 id 映射到连续向量。
- 位置编码或 RoPE 把顺序信息带进表示。

### 3. Computation layer

这里回答的是“模型内部如何算”。

- self-attention 负责让 token 彼此建立动态关系。
- multi-head 让模型在多个子空间并行观察上下文。
- RMSNorm 和 residual 负责稳定深层训练。
- FFN / SwiGLU 负责逐 token 的非线性加工。

### 4. Efficiency and capacity layer

这里回答的是“模型怎样更快、更大、更省”。

- GQA 通过减少 K/V head 数降低 cache 成本。
- MLA 继续压缩被缓存的状态表示。
- MoE 通过稀疏激活扩大总容量，而不让每个 token 都走满全部参数。

### 5. Training and alignment layer

这里回答的是“模型为什么不只是一个静态结构图”。

- pretraining 建立广义语言能力。
- continued training 把能力进一步往关键领域推。
- SFT 把模型推向更像助手的行为分布。
- reward model 和 preference optimization 再继续做偏好塑形。


In [ ]:
# 用一个结构化蓝图，把前面的模块重新串起来。
# 这里不是正式配置文件，而是帮助建立“整个系统由哪些层组成”的总图。

model_blueprint = {
    'data_pipeline': [
        'crawl and collect',
        'clean and filter',
        'deduplicate and bucket',
    ],
    'input_pipeline': [
        'tokenizer',
        'token embedding',
        'position encoding or RoPE',
    ],
    'backbone': {
        'block_structure': 'RMSNorm -> Attention -> Residual -> RMSNorm -> FFN -> Residual',
        'attention_variants': ['MHA', 'MQA', 'GQA', 'MLA'],
        'ffn_variants': ['Dense FFN', 'SwiGLU', 'MoE'],
    },
    'training_pipeline': [
        'pretraining',
        'continued training',
        'sft',
        'reward model',
        'ppo / dpo / grpo',
    ],
}

for key, value in model_blueprint.items():
    print(key, '=>', value)


## Trace one request through the whole system

如果一个用户现在问模型一句话，从系统角度看，大致会经过下面这些步骤：

1. 文本先被 tokenizer 切成 token ids。
2. token ids 进入 embedding table，变成向量。
3. 位置编码或 RoPE 让模型知道顺序。
4. 一层层 Transformer block 反复更新表示。
5. attention 在每一层里决定“当前位置该看谁”。
6. FFN / SwiGLU 在 token 内部继续加工特征。
7. 如果是 GQA / MLA 结构，推理时还会特别考虑 cache 的存储方式。
8. 最后输出 logits，对下一 token 给出概率分布。
9. 解码策略再从这个分布里选出实际输出。

注意，这只是**推理当下**发生的链路。而模型为什么会具备某种回答风格，则又要回到训练线去看。


In [ ]:
# 下面这个 toy trace 不做真正神经网络计算，重点是把 shape 和阶段串起来。

trace = {
    'raw_text': 'Explain what GQA solves.',
    'token_ids_shape': '(seq_len,)',
    'embedding_shape': '(seq_len, d_model)',
    'attention_input_shape': '(seq_len, d_model)',
    'logits_shape': '(seq_len, vocab_size)',
    'next_token_distribution_shape': '(vocab_size,)',
}

for name, value in trace.items():
    print(f'{name}: {value}')


## Where each advanced module fits

前面很多新名词，如果单独看会显得散。现在把它们放回“它们在解决什么瓶颈”这个问题里，就会清楚很多。

- **BPE**：解决的是文本离散化和词表构造问题。
- **RoPE**：解决的是 attention 缺乏顺序感的问题。
- **GQA**：主要解决推理时 K/V cache 太贵的问题。
- **MLA**：在 GQA 之外，继续从缓存表示层面压缩推理成本。
- **MoE**：主要解决“想做更大容量，但不想每个 token 都经过全部参数”的问题。
- **SFT**：解决 base model 虽强但不够像助手的问题。
- **Reward model / RLHF**：解决“会回答”和“更符合偏好地回答”之间的差距。

真正成熟的阅读方式，不是看见一个新词就单独记定义，而是先问：**它到底在补哪一个系统瓶颈？**


In [ ]:
# 用一个“模块 -> 主要瓶颈”的映射，强化这种阅读方式。

module_to_bottleneck = {
    'BPE': 'efficient text discretization and vocabulary construction',
    'RoPE': 'injecting position into attention geometry',
    'GQA': 'reducing KV cache size and bandwidth cost',
    'MLA': 'compressing cached attention state further',
    'MoE': 'expanding total capacity with sparse activation',
    'SFT': 'shaping instruction-following behavior',
    'Reward model': 'turning preference comparisons into trainable scores',
}

for module, bottleneck in module_to_bottleneck.items():
    print(f'{module}: {bottleneck}')


## The DeepSeek-like closed loop

如果把这个项目放回 DeepSeek-like 路线，一个更完整的闭环大致是：

1. **先有数据工程**：收集、清洗、去重、质量控制。
2. **再有基础结构**：tokenizer、Transformer、RoPE、FFN、norm、residual。
3. **再有效率与容量设计**：GQA、MLA、MoE 这类结构改造。
4. **再有多阶段训练**：pretraining -> continued training -> SFT -> reward model -> RLHF / PPO / GRPO。
5. **最后才是实际产品行为**：模型在推理时表现出来的风格、速度、长度能力和帮助性，都是这些环节共同作用的结果。

所以现代 LLM 从来都不是“只看结构图”或者“只看训练数据”就能理解的系统。你必须把结构、数据、训练三条线一起看。


## Summary

如果前面的 notebook 都真正跟下来，到这里你应该已经能做到三件事：

1. 从输入表示角度，讲清楚文本如何一步步进入模型。
2. 从结构角度，讲清楚 attention、GQA、MLA、MoE 分别在解决什么问题。
3. 从训练角度，讲清楚 pretraining、SFT、reward model、RLHF / PPO / GRPO 是怎样接力塑造模型的。

这就是这个项目最核心的产出：不是背了一堆术语，而是形成了一张结构化地图。以后你再去读 DeepSeek、LLaMA、Qwen 或其他现代 LLM 论文时，新的设计就不再是漂在空中的名词，而是会自然落回这张地图里的某一个位置。
